
# Agentic AI Workflow and Tool Integration


In [5]:

import pandas as pd
import numpy as np
import joblib
from pathlib import Path

# Define project directories
DATA_DIR = Path("../data")
PROCESSED_DIR = DATA_DIR / "processed"
MODEL_DIR = Path("../models")

# Load trained forecasting model
model_package = joblib.load(
    MODEL_DIR / "horizon_lightgbm.pkl"
)

forecast_model = model_package["model"]
feature_columns = model_package["feature_columns"]

# Load inventory recommendations
inventory_recommendations = pd.read_csv(
    PROCESSED_DIR / "inventory_recommendations.csv"
)

# Load business decision plan
business_decisions = pd.read_csv(
    PROCESSED_DIR / "business_decision_plan.csv"
)

# Verify
print("Agentic AI environment ready!")
print("Forecasting features:", len(feature_columns))
print("Inventory recommendations:", len(inventory_recommendations))
print("Business decisions:", len(business_decisions))

print(
    "Pending manager decisions:",
    (business_decisions["decision_status"] == "PENDING").sum()
)


Agentic AI environment ready!
Forecasting features: 10
Inventory recommendations: 273
Business decisions: 273
Pending manager decisions: 273



### 2. Build an Inventory Recommendation Lookup Tool


In [6]:

def get_inventory_recommendation(item_id):

    # Find the requested product
    result = inventory_recommendations.loc[
        inventory_recommendations["item_id"] == item_id
    ]

    # Handle products without a reorder recommendation
    if result.empty:
        return {
            "item_id": item_id,
            "status": "NOT_FOUND",
            "message": "No pending reorder recommendation found."
        }

    # Get the product's information
    product = result.iloc[0]

    return {
        "item_id": str(product["item_id"]),
        "priority": str(product["priority"]),
        "recommended_order_qty": int(
            product["recommended_order_qty"]
        ),
        "inventory_position": int(
            product["inventory_position"]
        ),
        "reorder_point": int(
            product["reorder_point"]
        ),
        "approval_status": str(
            product["approval_status"]
        )
    }


# Test our first tool
result = get_inventory_recommendation("FOODS_3_090")

print(result)


{'item_id': 'FOODS_3_090', 'priority': 'HIGH', 'recommended_order_qty': 744, 'inventory_position': 56, 'reorder_point': 339, 'approval_status': 'PENDING'}



### 3. Build a Business Risk Lookup Tool


In [7]:

def get_business_risk(item_id):

    # Find the product in our business decision plan
    result = business_decisions.loc[
        business_decisions["item_id"] == item_id
    ]

    if result.empty:
        return {
            "item_id": item_id,
            "status": "NOT_FOUND",
            "message": "No business decision record found."
        }

    product = result.iloc[0]

    return {
        "item_id": str(product["item_id"]),
        "estimated_order_cost": round(
            float(product["estimated_order_cost"]), 2
        ),
        "allocated_order_qty": int(
            product["new_allocated_qty"]
        ),
        "allocated_order_cost": round(
            float(product["new_allocated_cost"]), 2
        ),
        "shortage_severity": str(
            product["shortage_severity"]
        ),
        "management_action": str(
            product["management_action"]
        ),
        "decision_status": str(
            product["decision_status"]
        )
    }


# Test our business tool
result = get_business_risk("FOODS_3_090")

print(result)


{'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}



### 4. Build a Coordinator for Inventory and Business Tools


In [8]:

def coordinate_product_analysis(item_id):

    # Ask the inventory tool
    inventory_info = get_inventory_recommendation(item_id)

    # Ask the business tool
    business_info = get_business_risk(item_id)

    # Check that both tools found the product
    if (
        inventory_info.get("status") == "NOT_FOUND"
        or business_info.get("status") == "NOT_FOUND"
    ):
        return {
            "item_id": item_id,
            "status": "INCOMPLETE_DATA",
            "message": "Product information is unavailable in one or more tools."
        }

    # Combine both results
    report = {
        "item_id": item_id,
        "inventory": inventory_info,
        "business": business_info,
        "overall_status": "AWAITING_MANAGER_REVIEW"
    }

    return report


# Test the coordinator
coordinator_result = coordinate_product_analysis(
    "FOODS_3_090"
)

from pprint import pprint
pprint(coordinator_result)


{'business': {'allocated_order_cost': 0.0,
              'allocated_order_qty': 0,
              'decision_status': 'PENDING',
              'estimated_order_cost': 14560.08,
              'item_id': 'FOODS_3_090',
              'management_action': 'REVIEW_DELIVERY_SCHEDULE',
              'shortage_severity': 'MODERATE'},
 'inventory': {'approval_status': 'PENDING',
               'inventory_position': 56,
               'item_id': 'FOODS_3_090',
               'priority': 'HIGH',
               'recommended_order_qty': 744,
               'reorder_point': 339},
 'item_id': 'FOODS_3_090',
 'overall_status': 'AWAITING_MANAGER_REVIEW'}



### 5. Detect Inventory and Budget Conflicts


In [9]:

def detect_business_conflicts(item_id):

    # Get the combined report
    report = coordinate_product_analysis(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    inventory_info = report["inventory"]
    business_info = report["business"]

    conflicts = []

    # Check whether an urgent product has no funding
    if (
        inventory_info["priority"] == "HIGH"
        and business_info["allocated_order_qty"] == 0
    ):
        conflicts.append("HIGH_PRIORITY_WITHOUT_FUNDING")

    # Check whether the order is only partially funded
    if (
        0 < business_info["allocated_order_qty"]
        < inventory_info["recommended_order_qty"]
    ):
        conflicts.append("PARTIAL_FUNDING")

    # Check for serious delivery risk
    if business_info["shortage_severity"] == "SEVERE":
        conflicts.append("SEVERE_SHORTAGE_RISK")

    return {
        "item_id": item_id,
        "conflicts": conflicts,
        "conflict_count": len(conflicts),
        "requires_manager_review": len(conflicts) > 0
    }


# Test conflict detection
conflict_result = detect_business_conflicts(
    "FOODS_3_090"
)

print(conflict_result)


{'item_id': 'FOODS_3_090', 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'], 'conflict_count': 1, 'requires_manager_review': True}



### 6. Generate Management Actions from Business Conflicts


In [10]:

def suggest_management_actions(item_id):

    # Find conflicts using our existing tool
    result = detect_business_conflicts(item_id)

    if result.get("status") == "INCOMPLETE_DATA":
        return result

    # Define actions for each type of conflict
    action_rules = {
        "HIGH_PRIORITY_WITHOUT_FUNDING": (
            "Review budget allocation or explore supplier payment terms."
        ),
        "PARTIAL_FUNDING": (
            "Review whether the funded quantity provides enough coverage."
        ),
        "SEVERE_SHORTAGE_RISK": (
            "Check faster delivery options and existing incoming orders."
        )
    }

    # Match conflicts with management suggestions
    actions = [
        action_rules[conflict]
        for conflict in result["conflicts"]
        if conflict in action_rules
    ]

    if not actions:
        actions = [
            "Continue standard inventory monitoring and manager review."
        ]

    return {
        "item_id": item_id,
        "conflicts": result["conflicts"],
        "suggested_actions": actions,
        "approval_status": "PENDING"
    }


# Test our action suggestion tool
action_result = suggest_management_actions(
    "FOODS_3_090"
)

from pprint import pprint
pprint(action_result)


{'approval_status': 'PENDING',
 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
 'item_id': 'FOODS_3_090',
 'suggested_actions': ['Review budget allocation or explore supplier payment '
                       'terms.']}



### 7. Generate a Complete Product Decision Report


In [11]:

def generate_product_decision_report(item_id):

    # Get inventory and financial information
    report = coordinate_product_analysis(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    # Get detected conflicts and suggested actions
    actions = suggest_management_actions(item_id)

    inventory = report["inventory"]
    business = report["business"]

    # Build one complete report
    return {
        "item_id": item_id,
        "priority": inventory["priority"],
        "recommended_order_qty": inventory["recommended_order_qty"],
        "estimated_order_cost": business["estimated_order_cost"],
        "allocated_order_qty": business["allocated_order_qty"],
        "shortage_severity": business["shortage_severity"],
        "conflicts": actions["conflicts"],
        "suggested_actions": actions["suggested_actions"],
        "decision_status": "PENDING"
    }


# Test our complete report
final_report = generate_product_decision_report(
    "FOODS_3_090"
)

from pprint import pprint
pprint(final_report)


{'allocated_order_qty': 0,
 'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
 'decision_status': 'PENDING',
 'estimated_order_cost': 14560.08,
 'item_id': 'FOODS_3_090',
 'priority': 'HIGH',
 'recommended_order_qty': 744,
 'shortage_severity': 'MODERATE',
 'suggested_actions': ['Review budget allocation or explore supplier payment '
                       'terms.']}



### 8. Register Business Tools for Agent Coordination


In [12]:

# Register our existing business tools
tool_registry = {
    "inventory_lookup": {
        "function": get_inventory_recommendation,
        "description": "Retrieve product inventory and reorder information."
    },
    "business_risk_lookup": {
        "function": get_business_risk,
        "description": "Retrieve product costs, budget allocation and risks."
    },
    "conflict_detection": {
        "function": detect_business_conflicts,
        "description": "Identify inventory and business conflicts."
    },
    "management_actions": {
        "function": suggest_management_actions,
        "description": "Suggest management actions for business conflicts."
    },
    "decision_report": {
        "function": generate_product_decision_report,
        "description": "Generate a complete product decision report."
    }
}

# Check registered tools
print("Total registered tools:", len(tool_registry))

for name, details in tool_registry.items():
    print(f"{name}: {details['description']}")

# Test calling a registered tool
test_result = tool_registry["inventory_lookup"]["function"](
    "FOODS_3_090"
)

print("\nTool test:")
print(test_result["item_id"])
print("Recommended units:", test_result["recommended_order_qty"])


Total registered tools: 5
inventory_lookup: Retrieve product inventory and reorder information.
business_risk_lookup: Retrieve product costs, budget allocation and risks.
conflict_detection: Identify inventory and business conflicts.
management_actions: Suggest management actions for business conflicts.
decision_report: Generate a complete product decision report.

Tool test:
FOODS_3_090
Recommended units: 744



### 9. Build a Rule-Based Tool Router


In [13]:

def route_business_question(question, item_id):

    # Convert question to lowercase
    question = question.lower()

    # Choose a tool based on keywords
    if any(word in question for word in [
        "stock", "inventory", "reorder", "quantity"
    ]):
        tool_name = "inventory_lookup"

    elif any(word in question for word in [
        "cost", "budget", "afford", "price"
    ]):
        tool_name = "business_risk_lookup"

    elif any(word in question for word in [
        "conflict", "problem", "issue"
    ]):
        tool_name = "conflict_detection"

    elif any(word in question for word in [
        "action", "suggest", "recommendation"
    ]):
        tool_name = "management_actions"

    else:
        tool_name = "decision_report"

    # Call the selected tool
    result = tool_registry[tool_name]["function"](item_id)

    return {
        "question": question,
        "selected_tool": tool_name,
        "result": result
    }


# Test different questions
questions = [
    "How much stock should we reorder?",
    "What is the purchasing cost?",
    "Are there any business conflicts?",
    "What actions should management consider?"
]

for question in questions:
    response = route_business_question(
        question, "FOODS_3_090"
    )

    print("Question:", question)
    print("Selected tool:", response["selected_tool"])
    print()


Question: How much stock should we reorder?
Selected tool: inventory_lookup

Question: What is the purchasing cost?
Selected tool: business_risk_lookup

Question: Are there any business conflicts?
Selected tool: conflict_detection

Question: What actions should management consider?
Selected tool: management_actions




### 10. Build a Multi-Tool Business Router


In [14]:

def route_multiple_tools(question, item_id):

    question_lower = question.lower()
    selected_tools = []

    # Check inventory-related words
    if any(word in question_lower for word in [
        "stock", "inventory", "reorder", "quantity"
    ]):
        selected_tools.append("inventory_lookup")

    # Check financial-related words
    if any(word in question_lower for word in [
        "cost", "budget", "afford", "price"
    ]):
        selected_tools.append("business_risk_lookup")

    # Check conflict-related words
    if any(word in question_lower for word in [
        "conflict", "problem", "issue"
    ]):
        selected_tools.append("conflict_detection")

    # Check management-action words
    if any(word in question_lower for word in [
        "action", "suggest", "recommendation"
    ]):
        selected_tools.append("management_actions")

    # If no keywords match, use the full report
    if not selected_tools:
        selected_tools.append("decision_report")

    # Call each selected tool
    results = {}

    for tool_name in selected_tools:
        results[tool_name] = (
            tool_registry[tool_name]["function"](item_id)
        )

    return {
        "question": question,
        "item_id": item_id,
        "selected_tools": selected_tools,
        "tool_results": results
    }


# Test a question needing two tools
response = route_multiple_tools(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

print("Selected tools:", response["selected_tools"])

for tool_name, result in response["tool_results"].items():
    print(f"\n{tool_name}:")
    print(result)


Selected tools: ['inventory_lookup', 'business_risk_lookup']

inventory_lookup:
{'item_id': 'FOODS_3_090', 'priority': 'HIGH', 'recommended_order_qty': 744, 'inventory_position': 56, 'reorder_point': 339, 'approval_status': 'PENDING'}

business_risk_lookup:
{'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}



### 11. Generate Manager-Friendly Business Responses


In [15]:

def generate_manager_response(question, item_id):

    # Get information from relevant tools
    response = route_multiple_tools(question, item_id)

    results = response["tool_results"]
    messages = []

    # Explain inventory recommendations
    if "inventory_lookup" in results:
        info = results["inventory_lookup"]

        if info.get("status") == "NOT_FOUND":
            messages.append(info["message"])
        else:
            messages.append(
                f"Inventory: {info['item_id']} has "
                f"{info['inventory_position']} units in inventory position. "
                f"The proposed reorder is "
                f"{info['recommended_order_qty']} units "
                f"with {info['priority']} priority."
            )

    # Explain financial information
    if "business_risk_lookup" in results:
        info = results["business_risk_lookup"]

        if info.get("status") == "NOT_FOUND":
            messages.append(info["message"])
        else:
            messages.append(
                f"Budget: The full proposed order costs approximately "
                f"${info['estimated_order_cost']:,.2f}. "
                f"Funding was allocated for "
                f"{info['allocated_order_qty']} units."
            )

            if info["allocated_order_qty"] == 0:
                messages.append(
                    "No purchasing budget was allocated to this product. "
                    "Management should review funding before ordering."
                )

    # Handle tools that return other report formats
    for tool_name in response["selected_tools"]:
        if tool_name not in [
            "inventory_lookup",
            "business_risk_lookup"
        ]:
            messages.append(
                f"{tool_name}: {results[tool_name]}"
            )

    messages.append(
        "This is a simulated recommendation. "
        "Any purchasing decision requires manager approval."
    )

    return "\n\n".join(messages)


# Test the response
manager_answer = generate_manager_response(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

print(manager_answer)


Inventory: FOODS_3_090 has 56 units in inventory position. The proposed reorder is 744 units with HIGH priority.

Budget: The full proposed order costs approximately $14,560.08. Funding was allocated for 0 units.

No purchasing budget was allocated to this product. Management should review funding before ordering.

This is a simulated recommendation. Any purchasing decision requires manager approval.



### 12. Generate Structured Business Decision Responses


In [16]:

def generate_structured_response(item_id):

    # Get the complete business report
    report = generate_product_decision_report(item_id)

    if report.get("status") == "INCOMPLETE_DATA":
        return report

    # Organize the result into clear sections
    structured = {
        "product": {
            "item_id": report["item_id"],
            "priority": report["priority"]
        },

        "inventory": {
            "recommended_order_qty": report["recommended_order_qty"]
        },

        "finance": {
            "estimated_order_cost": report["estimated_order_cost"],
            "allocated_order_qty": report["allocated_order_qty"]
        },

        "risk": {
            "shortage_severity": report["shortage_severity"],
            "conflicts": report["conflicts"]
        },

        "management": {
            "suggested_actions": report["suggested_actions"],
            "decision_status": "PENDING"
        }
    }

    return structured


# Test structured response
structured_result = generate_structured_response(
    "FOODS_3_090"
)

from pprint import pprint
pprint(structured_result)


{'finance': {'allocated_order_qty': 0, 'estimated_order_cost': 14560.08},
 'inventory': {'recommended_order_qty': 744},
 'management': {'decision_status': 'PENDING',
                'suggested_actions': ['Review budget allocation or explore '
                                      'supplier payment terms.']},
 'product': {'item_id': 'FOODS_3_090', 'priority': 'HIGH'},
 'risk': {'conflicts': ['HIGH_PRIORITY_WITHOUT_FUNDING'],
          'shortage_severity': 'MODERATE'}}



### 13. Define Tools for LLM-Based Coordination


In [17]:

# Define tools using JSON-compatible descriptions
llm_tool_definitions = []

for name, details in tool_registry.items():
    llm_tool_definitions.append({
        "type": "function",
        "name": name,
        "description": details["description"],
        "parameters": {
            "type": "object",
            "properties": {
                "item_id": {
                    "type": "string",
                    "description": "The product ID to analyze."
                }
            },
            "required": ["item_id"],
            "additionalProperties": False
        },
        "strict": True
    })

# Verify tool definitions
print("Tools prepared for LLM:", len(llm_tool_definitions))

for tool in llm_tool_definitions:
    print(tool["name"])

print("\nFirst tool definition:")
from pprint import pprint
pprint(llm_tool_definitions[0])


Tools prepared for LLM: 5
inventory_lookup
business_risk_lookup
conflict_detection
management_actions
decision_report

First tool definition:
{'description': 'Retrieve product inventory and reorder information.',
 'name': 'inventory_lookup',
 'parameters': {'additionalProperties': False,
                'properties': {'item_id': {'description': 'The product ID to '
                                                          'analyze.',
                                           'type': 'string'}},
                'required': ['item_id'],
                'type': 'object'},
 'strict': True,
 'type': 'function'}



### 14. Check LLM SDK and API Configuration


In [18]:

import os
import importlib.util

# Check whether the OpenAI Python package is installed
sdk_installed = (
    importlib.util.find_spec("openai") is not None
)

# Check whether an API key is configured
# Never display the actual key
api_key_configured = bool(
    os.environ.get("OPENAI_API_KEY")
)

print("OpenAI SDK installed:", sdk_installed)
print("API key configured:", api_key_configured)
print("Registered business tools:", len(llm_tool_definitions))

if sdk_installed and api_key_configured:
    print("Environment ready for API integration.")
else:
    print("Additional setup needed before API integration.")


OpenAI SDK installed: True
API key configured: True
Registered business tools: 5
Environment ready for API integration.



### 15. Install and Verify the OpenAI Python SDK


In [19]:

%pip install -U openai


   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.4 MB ? eta -:--:--
   -------------------------------

In [20]:

import openai
import os

print("OpenAI SDK version:", openai.__version__)
print("API key configured:", bool(os.environ.get("OPENAI_API_KEY")))
print("SDK installation: PASSED")


OpenAI SDK version: 3.45.0
API key configured: True
SDK installation: PASSED



### 16. Initialize OpenAI API Client


In [21]:

import os
from openai import OpenAI

# Check that the API key exists
assert os.environ.get("OPENAI_API_KEY"), (
    "OPENAI_API_KEY is not configured."
)

# Initialize the OpenAI client
client = OpenAI(
    api_key=os.environ["OPENAI_API_KEY"]
)

# Verify the client without making an API request
print("OpenAI client initialized!")
print("API key available: True")
print("Responses API available:",
      hasattr(client, "responses"))
print("Registered business tools:", len(llm_tool_definitions))


OpenAI client initialized!
API key available: True
Responses API available: True
Registered business tools: 5



### 17. Simulate LLM Tool Selection Locally


In [22]:
import json

question = "Can we afford to purchase product FOODS_3_090?"
item_id = "FOODS_3_090"

# Use our existing rule-based router
local_result = route_business_question(
    question,
    item_id
)

# Simulate the tool-call format
simulated_tool_call = {
    "name": local_result["selected_tool"],
    "arguments": json.dumps({
        "item_id": item_id
    })
}

print("Mode: LOCAL SIMULATION")
print("Selected tool:", simulated_tool_call["name"])
print("Arguments:", simulated_tool_call["arguments"])
print("Tool selection simulated: True")
print("API requests made: 0")


Mode: LOCAL SIMULATION
Selected tool: business_risk_lookup
Arguments: {"item_id": "FOODS_3_090"}
Tool selection simulated: True
API requests made: 0



### 18. Execute a Registered Business Tool Safely


In [23]:

# Execute a tool selected by our local router
def execute_business_tool(tool_call):

    tool_name = tool_call["name"]

    # Only allow registered tools
    if tool_name not in tool_registry:
        return {
            "status": "BLOCKED",
            "reason": "Tool is not registered."
        }

    # Read and validate tool arguments
    try:
        arguments = json.loads(tool_call["arguments"])
    except (json.JSONDecodeError, TypeError):
        return {
            "status": "BLOCKED",
            "reason": "Invalid tool arguments."
        }

    # Only accept one non-empty product ID
    if (
        not isinstance(arguments, dict)
        or set(arguments) != {"item_id"}
        or not isinstance(arguments["item_id"], str)
        or not arguments["item_id"].strip()
    ):
        return {
            "status": "BLOCKED",
            "reason": "Invalid product ID."
        }

    # Execute the registered read-only function
    result = tool_registry[tool_name]["function"](
        arguments["item_id"]
    )

    return {
        "status": "SUCCESS",
        "tool": tool_name,
        "result": result
    }


# Execute our simulated tool call
execution_result = execute_business_tool(
    simulated_tool_call
)

from pprint import pprint
pprint(execution_result)


{'result': {'allocated_order_cost': 0.0,
            'allocated_order_qty': 0,
            'decision_status': 'PENDING',
            'estimated_order_cost': 14560.08,
            'item_id': 'FOODS_3_090',
            'management_action': 'REVIEW_DELIVERY_SCHEDULE',
            'shortage_severity': 'MODERATE'},
 'status': 'SUCCESS',
 'tool': 'business_risk_lookup'}



### 19. Test Tool Execution Safety Controls


In [24]:

# Test 1: An unregistered tool
unknown_tool = {
    "name": "send_payment",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090"
    })
}

# Test 2: Invalid JSON
invalid_json = {
    "name": "business_risk_lookup",
    "arguments": "{invalid_json}"
}

# Test 3: Unexpected extra argument
extra_argument = {
    "name": "business_risk_lookup",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090",
        "approve_order": True
    })
}

# Test 4: Valid registered tool
valid_tool = {
    "name": "inventory_lookup",
    "arguments": json.dumps({
        "item_id": "FOODS_3_090"
    })
}

tests = {
    "Unknown tool": unknown_tool,
    "Invalid JSON": invalid_json,
    "Unexpected argument": extra_argument,
    "Valid tool": valid_tool
}

for test_name, tool_call in tests.items():
    result = execute_business_tool(tool_call)

    print(f"{test_name}: {result['status']}")

    if result["status"] == "BLOCKED":
        print("Reason:", result["reason"])

    print()

# Verify expected behavior
assert execute_business_tool(unknown_tool)["status"] == "BLOCKED"
assert execute_business_tool(invalid_json)["status"] == "BLOCKED"
assert execute_business_tool(extra_argument)["status"] == "BLOCKED"
assert execute_business_tool(valid_tool)["status"] == "SUCCESS"

print("All four safety tests: PASSED")


Unknown tool: BLOCKED
Reason: Tool is not registered.

Invalid JSON: BLOCKED
Reason: Invalid tool arguments.

Unexpected argument: BLOCKED
Reason: Invalid product ID.

Valid tool: SUCCESS

All four safety tests: PASSED



### 20. Build a Complete Local Business Agent Workflow


In [25]:

def run_local_agent_workflow(question, item_id):

    # Step 1: Select tools using our local router
    routing = route_multiple_tools(question, item_id)

    # Step 2: Execute each tool through our safe dispatcher
    tool_results = {}

    for tool_name in routing["selected_tools"]:

        tool_call = {
            "name": tool_name,
            "arguments": json.dumps({"item_id": item_id})
        }

        execution = execute_business_tool(tool_call)

        tool_results[tool_name] = execution

    # Step 3: Return a consolidated report
    return {
        "question": question,
        "item_id": item_id,
        "mode": "LOCAL_RULE_BASED",
        "selected_tools": routing["selected_tools"],
        "tool_results": tool_results,
        "approval_status": "PENDING",
        "purchase_executed": False
    }


# Test the workflow
workflow_result = run_local_agent_workflow(
    "How much stock should we reorder, and can we afford it?",
    "FOODS_3_090"
)

from pprint import pprint
pprint(workflow_result)


{'approval_status': 'PENDING',
 'item_id': 'FOODS_3_090',
 'mode': 'LOCAL_RULE_BASED',
 'purchase_executed': False,
 'question': 'How much stock should we reorder, and can we afford it?',
 'selected_tools': ['inventory_lookup', 'business_risk_lookup'],
 'tool_results': {'business_risk_lookup': {'result': {'allocated_order_cost': 0.0,
                                                      'allocated_order_qty': 0,
                                                      'decision_status': 'PENDING',
                                                      'estimated_order_cost': 14560.08,
                                                      'item_id': 'FOODS_3_090',
                                                      'management_action': 'REVIEW_DELIVERY_SCHEDULE',
                                                      'shortage_severity': 'MODERATE'},
                                           'status': 'SUCCESS',
                                           'tool': 'business_risk_lookup'},
 


### 21. Create a Local Workflow Execution Log


In [26]:

from datetime import datetime, timezone

# Collect the workflow execution details
workflow_log = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "item_id": workflow_result["item_id"],
    "question": workflow_result["question"],
    "mode": workflow_result["mode"],
    "tools_used": workflow_result["selected_tools"],
    "tool_count": len(workflow_result["selected_tools"]),
    "all_tools_successful": all(
        result["status"] == "SUCCESS"
        for result in workflow_result["tool_results"].values()
    ),
    "approval_status": workflow_result["approval_status"],
    "purchase_executed": workflow_result["purchase_executed"]
}

# Verify the execution log
print("Product:", workflow_log["item_id"])
print("Execution mode:", workflow_log["mode"])
print("Tools used:", workflow_log["tools_used"])
print("Tool count:", workflow_log["tool_count"])
print("All tools successful:", workflow_log["all_tools_successful"])
print("Approval status:", workflow_log["approval_status"])
print("Purchase executed:", workflow_log["purchase_executed"])


Product: FOODS_3_090
Execution mode: LOCAL_RULE_BASED
Tools used: ['inventory_lookup', 'business_risk_lookup']
Tool count: 2
All tools successful: True
Approval status: PENDING
Purchase executed: False



### 22. Save and Verify Workflow Execution Log


In [27]:

import json
from pathlib import Path

# Create a folder for workflow logs
log_dir = Path("../data/processed/agent_logs")
log_dir.mkdir(parents=True, exist_ok=True)

# Define the log file
log_path = log_dir / "workflow_demo_log.json"

# Save the workflow log
with open(log_path, "w", encoding="utf-8") as f:
    json.dump(workflow_log, f, indent=4)

# Load it again to verify
with open(log_path, "r", encoding="utf-8") as f:
    saved_log = json.load(f)

# Verify the saved information
assert saved_log == workflow_log
assert saved_log["all_tools_successful"] is True
assert saved_log["purchase_executed"] is False

print("Workflow log saved successfully!")
print("File exists:", log_path.exists())
print("Log verification: PASSED")
print("Execution mode:", saved_log["mode"])
print("Purchase executed:", saved_log["purchase_executed"])


Workflow log saved successfully!
File exists: True
Log verification: PASSED
Execution mode: LOCAL_RULE_BASED
Purchase executed: False



### 23. Check Local LLM Hardware Requirements


In [28]:

import platform
import shutil
import subprocess

print("Operating system:", platform.system())
print("Processor:", platform.processor())
print("Ollama installed:", shutil.which("ollama") is not None)

# Check system RAM
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
    print(f"Total RAM: {ram_gb:.1f} GB")
except ImportError:
    print("RAM: Not detected (psutil unavailable)")

# Check NVIDIA GPU
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total",
         "--format=csv,noheader"],
        capture_output=True,
        text=True,
        timeout=10
    )

    if gpu.returncode == 0:
        print("GPU:", gpu.stdout.strip())
    else:
        print("NVIDIA GPU: Not detected")

except (FileNotFoundError, subprocess.TimeoutExpired):
    print("NVIDIA GPU: Not detected")


Operating system: Windows
Processor: AMD64 Family 26 Model 36 Stepping 0, AuthenticAMD
Ollama installed: True
Total RAM: 23.1 GB
NVIDIA GPU: Not detected



### 24. Check Available Local LLM Models


In [29]:

import subprocess

result = subprocess.run(
    ["ollama", "list"],
    capture_output=True,
    text=True,
    timeout=15
)

if result.returncode == 0:
    print("Ollama model list:")
    print(result.stdout)
else:
    print("Could not list models:")
    print(result.stderr)


Ollama model list:
NAME               ID              SIZE      MODIFIED      
llama3.2:latest    a80c4f17acd5    2.0 GB    13 months ago    
mistral:latest     3944fe81ec14    4.1 GB    16 months ago    
llama3:latest      365c0bd3c000    4.7 GB    17 months ago    




### 25. Test Local Llama 3.2 Model


In [30]:

import requests

# Connect to the local Ollama server
url = "http://localhost:11434/api/chat"

payload = {
    "model": "llama3.2:latest",
    "messages": [
        {
            "role": "user",
            "content": (
                "In one sentence, explain why a retail "
                "business needs inventory planning."
            )
        }
    ],
    "stream": False,
    "options": {
        "temperature": 0
    }
}

try:
    response = requests.post(
        url,
        json=payload,
        timeout=180
    )

    response.raise_for_status()

    result = response.json()

    print("Model:", result["model"])
    print("AI response:")
    print(result["message"]["content"])

except requests.exceptions.RequestException as e:
    print("Ollama connection error:", e)


Model: llama3.2:latest
AI response:
A retail business needs inventory planning to ensure that it has the right products in stock at the right time to meet customer demand, minimize stockouts and overstocking, and optimize inventory levels to maximize profitability and reduce waste.



### 26. Test Real LLM-Based Tool Selection


In [31]:

# Ollama requires a slightly different tool format
ollama_tools = []

for name, details in tool_registry.items():
    ollama_tools.append({
        "type": "function",
        "function": {
            "name": name,
            "description": details["description"],
            "parameters": {
                "type": "object",
                "properties": {
                    "item_id": {
                        "type": "string",
                        "description": "Product ID to analyze."
                    }
                },
                "required": ["item_id"]
            }
        }
    })

# Give the LLM our five available tools
payload = {
    "model": "llama3.2:latest",
    "messages": [
        {
            "role": "system",
            "content": (
                "You are an intelligent retail business assistant. "
                "Use the provided tools to retrieve business data. "
                "Never invent inventory values or financial numbers. "
                "Never approve or execute purchases."
            )
        },
        {
            "role": "user",
            "content": (
                "How much inventory should we reorder "
                "for product FOODS_3_090?"
            )
        }
    ],
    "tools": ollama_tools,
    "stream": False,
    "options": {"temperature": 0}
}

response = requests.post(
    "http://localhost:11434/api/chat",
    json=payload,
    timeout=180
)

response.raise_for_status()
llm_result = response.json()

tool_calls = llm_result["message"].get(
    "tool_calls", []
)

print("Model:", llm_result["model"])
print("Number of tool calls:", len(tool_calls))

for call in tool_calls:
    print("Selected tool:", call["function"]["name"])
    print("Arguments:", call["function"]["arguments"])

print("Tools executed: 0")


Model: llama3.2:latest
Number of tool calls: 1
Selected tool: inventory_lookup
Arguments: {'item_id': 'FOODS_3_090'}
Tools executed: 0



### 27. Execute LLM-Requested Business Tool


In [32]:

import json
from pprint import pprint

# Collect the LLM's requested tool calls
llm_tool_results = []

for call in tool_calls:

    tool_name = call["function"]["name"]
    arguments = call["function"]["arguments"]

    # Ollama may return arguments as a dictionary
    if isinstance(arguments, dict):
        arguments = json.dumps(arguments)

    # Reuse our validated tool executor
    execution = execute_business_tool({
        "name": tool_name,
        "arguments": arguments
    })

    llm_tool_results.append(execution)

    print("LLM selected:", tool_name)
    print("Execution status:", execution["status"])
    print("Business tool result:")
    pprint(execution.get("result", execution.get("reason")))


LLM selected: inventory_lookup
Execution status: SUCCESS
Business tool result:
{'approval_status': 'PENDING',
 'inventory_position': 56,
 'item_id': 'FOODS_3_090',
 'priority': 'HIGH',
 'recommended_order_qty': 744,
 'reorder_point': 339}



### 28. Generate an LLM Response Using Tool Results


In [33]:

# Preserve the conversation from Step 26
messages = payload["messages"].copy()

# Add the LLM's tool request
messages.append(llm_result["message"])

# Add the actual tool results from Step 27
for call, execution in zip(tool_calls, llm_tool_results):
    messages.append({
        "role": "tool",
        "tool_name": call["function"]["name"],
        "content": json.dumps(execution)
    })

# Ask Llama 3.2 to explain the results
final_payload = {
    "model": "llama3.2:latest",
    "messages": messages,
    "stream": False,
    "options": {"temperature": 0}
}

final_response = requests.post(
    "http://localhost:11434/api/chat",
    json=final_payload,
    timeout=180
)

final_response.raise_for_status()

final_result = final_response.json()

print("AI Manager's Answer:")
print(final_result["message"]["content"])

print("\nAdditional tool calls requested:",
      len(final_result["message"].get("tool_calls", [])))


AI Manager's Answer:
Based on the inventory lookup result, it is recommended to reorder 744 units of product FOODS_3_090. The current inventory position is 56, and the recommended reorder point is 339. Please note that this information is subject to change and may require further review or approval.

Additional tool calls requested: 0



### 29. Test LLM Multi-Tool Selection


In [34]:

# Ask a question that requires inventory and finance data
multi_tool_question = (
    "For product FOODS_3_090, how many units should "
    "we reorder, what will they cost, and how much "
    "funding has been allocated?"
)

multi_tool_messages = [
    {
        "role": "system",
        "content": (
            "You are an intelligent retail business assistant. "
            "Choose all relevant tools needed to answer the question. "
            "Use inventory_lookup for stock and reorder quantities. "
            "Use business_risk_lookup for costs and budget allocation. "
            "Never invent business numbers. "
            "Never approve or execute purchases."
        )
    },
    {
        "role": "user",
        "content": multi_tool_question
    }
]

multi_tool_payload = {
    "model": "llama3.2:latest",
    "messages": multi_tool_messages,
    "tools": ollama_tools,
    "stream": False,
    "options": {
        "temperature": 0
    }
}

# Ask the real local LLM
multi_response = requests.post(
    "http://localhost:11434/api/chat",
    json=multi_tool_payload,
    timeout=180
)

multi_response.raise_for_status()
multi_result = multi_response.json()

# Inspect the LLM's requested tools
multi_tool_calls = multi_result["message"].get(
    "tool_calls", []
)

selected_names = [
    call["function"]["name"]
    for call in multi_tool_calls
]

print("Model:", multi_result["model"])
print("Number of tool calls:", len(multi_tool_calls))
print("Selected tools:", selected_names)

print(
    "Inventory tool selected:",
    "inventory_lookup" in selected_names
)
print(
    "Finance tool selected:",
    "business_risk_lookup" in selected_names
)

print("Tools executed: 0")


Model: llama3.2:latest
Number of tool calls: 2
Selected tools: ['inventory_lookup', 'business_risk_lookup']
Inventory tool selected: True
Finance tool selected: True
Tools executed: 0



### 30. Execute Multiple LLM-Selected Business Tools


In [35]:

import json
from pprint import pprint

# Store results from both tools
multi_tool_results = []

for call in multi_tool_calls:

    tool_name = call["function"]["name"]
    arguments = call["function"]["arguments"]

    # Convert Ollama's arguments into JSON if needed
    if isinstance(arguments, dict):
        arguments = json.dumps(arguments)

    # Execute through our validated, read-only dispatcher
    execution = execute_business_tool({
        "name": tool_name,
        "arguments": arguments
    })

    multi_tool_results.append(execution)

    print("Tool:", tool_name)
    print("Status:", execution["status"])

    if execution["status"] == "SUCCESS":
        pprint(execution["result"])
    else:
        print("Reason:", execution["reason"])

    print("-" * 40)

# Check that both expected tools ran successfully
successful_tools = {
    result["tool"]
    for result in multi_tool_results
    if result["status"] == "SUCCESS"
}

expected_tools = {
    "inventory_lookup",
    "business_risk_lookup"
}

print(
    "Both required tools successful:",
    expected_tools.issubset(successful_tools)
)
print("Purchases executed: 0")


Tool: inventory_lookup
Status: SUCCESS
{'approval_status': 'PENDING',
 'inventory_position': 56,
 'item_id': 'FOODS_3_090',
 'priority': 'HIGH',
 'recommended_order_qty': 744,
 'reorder_point': 339}
----------------------------------------
Tool: business_risk_lookup
Status: SUCCESS
{'allocated_order_cost': 0.0,
 'allocated_order_qty': 0,
 'decision_status': 'PENDING',
 'estimated_order_cost': 14560.08,
 'item_id': 'FOODS_3_090',
 'management_action': 'REVIEW_DELIVERY_SCHEDULE',
 'shortage_severity': 'MODERATE'}
----------------------------------------
Both required tools successful: True
Purchases executed: 0



### 31. Generate an LLM Recommendation from Multiple Tool Results


In [36]:

# Keep the original question and LLM tool requests
manager_messages = multi_tool_messages.copy()
manager_messages.append(multi_result["message"])

# Return both tool results to the LLM
for call, execution in zip(multi_tool_calls, multi_tool_results):
    manager_messages.append({
        "role": "tool",
        "tool_name": call["function"]["name"],
        "content": json.dumps(execution)
    })

# Ask Llama to explain the combined results
manager_payload = {
    "model": "llama3.2:latest",
    "messages": manager_messages,
    "stream": False,
    "options": {"temperature": 0}
}

manager_response = requests.post(
    "http://localhost:11434/api/chat",
    json=manager_payload,
    timeout=180
)

manager_response.raise_for_status()
manager_result = manager_response.json()

print("AI Manager's Recommendation:")
print(manager_result["message"]["content"])

print(
    "\nAdditional tool calls:",
    len(manager_result["message"].get("tool_calls", []))
)


AI Manager's Recommendation:
Based on the inventory_lookup and business_risk_lookup tools, here is the information requested:

* Reorder quantity: The recommended reorder quantity for FOODS_3_090 is 744 units.
* Cost: The estimated order cost for FOODS_3_090 is $14560.08.

However, I couldn't find any allocated funding for this item in the business_risk_lookup tool. It's recommended to review and manage the delivery schedule to mitigate potential shortages.

Additional tool calls: 0



### 32. Build a Bounded Local LLM Agent Loop


In [37]:

def run_ollama_agent(question, max_rounds=4):

    messages = [
        {
            "role": "system",
            "content": (
                "You are a retail business analysis agent. "
                "Use the available tools to obtain factual business data. "
                "Do not invent inventory numbers, financial values, "
                "or approval decisions. "
                "Clearly distinguish budget allocation from total "
                "company funds. "
                "Explain shortage risks and approval status. "
                "All purchase recommendations require human approval. "
                "Never claim an order has been placed."
            )
        },
        {"role": "user", "content": question}
    ]

    execution_log = []

    for round_number in range(1, max_rounds + 1):

        response = requests.post(
            "http://localhost:11434/api/chat",
            json={
                "model": "llama3.2:latest",
                "messages": messages,
                "tools": ollama_tools,
                "stream": False,
                "options": {"temperature": 0}
            },
            timeout=180
        )

        response.raise_for_status()
        assistant_message = response.json()["message"]
        messages.append(assistant_message)

        calls = assistant_message.get("tool_calls", [])

        if not calls:
            return {
                "status": "COMPLETED",
                "answer": assistant_message.get("content", ""),
                "rounds": round_number,
                "execution_log": execution_log,
                "purchase_executed": False
            }

        # Limit how many tools execute in one round
        for call in calls[:5]:

            name = call["function"]["name"]
            arguments = call["function"]["arguments"]

            if isinstance(arguments, dict):
                arguments = json.dumps(arguments)

            execution = execute_business_tool({
                "name": name,
                "arguments": arguments
            })

            
            execution_log.append({
                "round": round_number,
                "tool": name,
                "status": execution["status"],
                "result": execution.get("result"),
                "error": execution.get("reason")
            })


            messages.append({
                "role": "tool",
                "tool_name": name,
                "content": json.dumps(execution)
            })

        print(f"Round {round_number}: {len(calls[:5])} tool(s) executed")

    return {
        "status": "MAX_ROUNDS_REACHED",
        "answer": None,
        "rounds": max_rounds,
        "execution_log": execution_log,
        "purchase_executed": False
    }


# Test our agent
agent_result = run_ollama_agent(
    "For product FOODS_3_090, how much should we "
    "reorder, what is the cost, and what business "
    "risks should the manager review?"
)

print("\nStatus:", agent_result["status"])
print("Rounds:", agent_result["rounds"])
print("Execution log:", agent_result["execution_log"])
print("\nAgent answer:")
print(agent_result["answer"])
print("\nPurchase executed:", agent_result["purchase_executed"])


Round 1: 1 tool(s) executed

Status: COMPLETED
Rounds: 2
Execution log: [{'round': 1, 'tool': 'business_risk_lookup', 'status': 'SUCCESS', 'result': {'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}, 'error': None}]

Agent answer:
For the product FOODS_3_090, the recommended reordering quantity is not explicitly stated in the output as allocated_order_qty is 0. However, based on the estimated order cost of $14,560.08, a reasonable reordering quantity could be determined by analyzing historical demand and lead time.

The business risks that the manager should review for FOODS_3_090 are:

1. **Shortage Severity**: Moderate - This indicates that there is a moderate risk of stockouts, and the manager should ensure that delivery schedules are reviewed to minimize this risk.
2. **Management Action**: Review Delivery 


### 33. Improve LLM Tool Descriptions


In [38]:

# Make the tool descriptions more specific
tool_descriptions = {
    "inventory_lookup": (
        "Get the recommended reorder quantity, current inventory "
        "position, reorder point, priority, and approval status. "
        "ALWAYS use this tool when the question asks how many units "
        "to order, reorder, purchase, or restock."
    ),

    "business_risk_lookup": (
        "Get estimated purchasing cost, allocated funding, "
        "shortage severity, and management actions. "
        "Use for budget, cost, financial risk, or shortage questions. "
        "IMPORTANT: allocated_order_qty is the quantity funded "
        "under the budget plan, NOT the recommended reorder quantity."
    ),

    "conflict_detection": (
        "Identify conflicts between inventory needs, "
        "funding allocation, and shortage risks."
    ),

    "management_actions": (
        "Suggest management actions for identified business conflicts."
    ),

    "decision_report": (
        "Get a combined product decision report covering inventory, "
        "finance, conflicts, and management recommendations."
    )
}

# Update the Ollama tool definitions
for tool in ollama_tools:
    name = tool["function"]["name"]
    tool["function"]["description"] = tool_descriptions[name]

print("Updated tool descriptions:", len(ollama_tools))

for tool in ollama_tools:
    print(
        tool["function"]["name"],
        "—",
        tool["function"]["description"][:75] + "..."
    )


Updated tool descriptions: 5
inventory_lookup — Get the recommended reorder quantity, current inventory position, reorder p...
business_risk_lookup — Get estimated purchasing cost, allocated funding, shortage severity, and ma...
conflict_detection — Identify conflicts between inventory needs, funding allocation, and shortag...
management_actions — Suggest management actions for identified business conflicts....
decision_report — Get a combined product decision report covering inventory, finance, conflic...



### 34. Retest Agent with Improved Tool Descriptions


In [39]:

# Ask the exact same question as Step 32
agent_result_v2 = run_ollama_agent(
    "For product FOODS_3_090, how much should we "
    "reorder, what is the cost, and what business "
    "risks should the manager review?"
)

print("\n--- AGENT EVALUATION ---")

tools_used = [
    entry["tool"]
    for entry in agent_result_v2["execution_log"]
    if entry["status"] == "SUCCESS"
]

print("Agent status:", agent_result_v2["status"])
print("Tools used:", tools_used)

print(
    "Inventory tool used:",
    "inventory_lookup" in tools_used
)

print(
    "Finance tool used:",
    "business_risk_lookup" in tools_used
)

print("\n--- FINAL ANSWER ---")
print(agent_result_v2["answer"])

print(
    "\nPurchase executed:",
    agent_result_v2["purchase_executed"]
)


Round 1: 1 tool(s) executed

--- AGENT EVALUATION ---
Agent status: COMPLETED
Tools used: ['business_risk_lookup']
Inventory tool used: False
Finance tool used: True

--- FINAL ANSWER ---
For the product FOODS_3_090, the recommended reordering quantity is not explicitly stated in the output as allocated_order_qty is 0. However, based on the estimated order cost of $14,560.08, a reasonable reordering quantity could be determined by analyzing historical demand and lead time.

The business risks that the manager should review for FOODS_3_090 are:

1. **Shortage Severity**: Moderate - This indicates that there is a moderate risk of stockouts, and the manager should ensure that delivery schedules are reviewed to minimize this risk.
2. **Management Action**: Review Delivery Schedule - The manager should review the delivery schedule to ensure that it aligns with the company's inventory management strategy and minimizes the risk of stockouts.

Please note that these recommendations require hum


### 35. Validate Agent Evidence Completeness


In [40]:

def validate_agent_evidence(agent_result, required_tools):

    # Collect successfully executed tools
    successful_tools = {
        entry["tool"]
        for entry in agent_result["execution_log"]
        if entry["status"] == "SUCCESS"
    }

    # Identify missing evidence sources
    missing_tools = [
        tool
        for tool in required_tools
        if tool not in successful_tools
    ]

    return {
        "agent_status": agent_result["status"],
        "successful_tools": sorted(successful_tools),
        "missing_tools": missing_tools,
        "evidence_complete": (
            agent_result["status"] == "COMPLETED"
            and len(missing_tools) == 0
        ),
        "manager_approval_required": True
    }


# Define the evidence required for this question
required_tools = [
    "inventory_lookup",
    "business_risk_lookup"
]

# Validate the response from Step 34
evidence_check = validate_agent_evidence(
    agent_result_v2,
    required_tools
)

from pprint import pprint
pprint(evidence_check)


{'agent_status': 'COMPLETED',
 'evidence_complete': False,
 'manager_approval_required': True,
 'missing_tools': ['inventory_lookup'],
 'successful_tools': ['business_risk_lookup']}



### 36. Recover Missing Business Evidence


In [41]:

import json

def recover_missing_evidence(evidence_check, item_id):

    recovery_results = {}

    for tool_name in evidence_check["missing_tools"]:

        # Execute only through our approved dispatcher
        execution = execute_business_tool({
            "name": tool_name,
            "arguments": json.dumps({
                "item_id": item_id
            })
        })

        recovery_results[tool_name] = execution

    return recovery_results


# Retrieve the missing inventory information
recovery_results = recover_missing_evidence(
    evidence_check,
    "FOODS_3_090"
)

# Display recovery results
for tool_name, execution in recovery_results.items():

    print("Recovered tool:", tool_name)
    print("Execution status:", execution["status"])

    if execution["status"] == "SUCCESS":
        print("Recommended order:",
              execution["result"].get("recommended_order_qty"))
        print("Inventory position:",
              execution["result"].get("inventory_position"))

# Verify that all required evidence is now available
recovered_successfully = all(
    result["status"] == "SUCCESS"
    for result in recovery_results.values()
)

print("Missing evidence recovered:", recovered_successfully)
print("Purchases executed: 0")


Recovered tool: inventory_lookup
Execution status: SUCCESS
Recommended order: 744
Inventory position: 56
Missing evidence recovered: True
Purchases executed: 0



### 37. Generate and Validate a Corrected AI Recommendation


In [42]:

# Retrieve both verified business results
inventory_evidence = recovery_results["inventory_lookup"]
finance_evidence = execute_business_tool({
    "name": "business_risk_lookup",
    "arguments": json.dumps({"item_id": "FOODS_3_090"})
})

assert inventory_evidence["status"] == "SUCCESS"
assert finance_evidence["status"] == "SUCCESS"

# Create a clear evidence package
evidence_package = {
    "inventory": inventory_evidence["result"],
    "finance": finance_evidence["result"]
}

# Ask the local LLM to explain verified evidence
corrected_payload = {
    "model": "llama3.2:latest",
    "messages": [
        {
            "role": "system",
            "content": (
                "You are a retail decision-support assistant. "
                "Use only the supplied evidence. "
                "Report the recommended reorder quantity, estimated "
                "cost, allocated funding, shortage severity, and "
                "approval status. "
                "Do not confuse funded quantity with reorder quantity. "
                "Do not invent numbers or claim purchases were made."
            )
        },
        {
            "role": "user",
            "content": (
                "Explain the purchasing recommendation for "
                "FOODS_3_090 using this verified evidence:\n"
                + json.dumps(evidence_package)
            )
        }
    ],
    "stream": False,
    "options": {"temperature": 0}
}

corrected_response = requests.post(
    "http://localhost:11434/api/chat",
    json=corrected_payload,
    timeout=180
)

corrected_response.raise_for_status()

corrected_answer = corrected_response.json()["message"]["content"]

print("CORRECTED AI RECOMMENDATION:")
print(corrected_answer)

# Basic text-based checks
answer_lower = corrected_answer.lower()

checks = {
    "Reorder quantity mentioned": "744" in corrected_answer,
    "Estimated cost mentioned": (
        "14,560.08" in corrected_answer
        or "14560.08" in corrected_answer
    ),
    "Shortage risk mentioned": "moderate" in answer_lower,
    "Approval mentioned": (
        "pending" in answer_lower
        or "approval" in answer_lower
    )
}

print("\n--- BASIC ANSWER CHECKS ---")
for name, passed in checks.items():
    print(f"{name}: {passed}")

print("All checks passed:", all(checks.values()))


CORRECTED AI RECOMMENDATION:
Based on the provided evidence, here is the purchasing recommendation for FOODS_3_090:

- Recommended Reorder Quantity: 744
- Estimated Cost: $14,560.08 per order
- Allocated Funding: Not specified (allocated_order_qty and allocated_order_cost are both 0)
- Shortage Severity: MODERATE
- Approval Status: PENDING

--- BASIC ANSWER CHECKS ---
Reorder quantity mentioned: True
Estimated cost mentioned: True
Shortage risk mentioned: True
Approval mentioned: True
All checks passed: True



### 38. Validate Business Facts Against Tool Evidence


In [43]:

# Read authoritative values from our existing tools
inventory_data = evidence_package["inventory"]
finance_data = evidence_package["finance"]

# Check consistency between the two tools
assert inventory_data["item_id"] == finance_data["item_id"]

# Build verified facts directly from tool results
verified_facts = {
    "item_id": inventory_data["item_id"],
    "recommended_order_qty": inventory_data["recommended_order_qty"],
    "estimated_order_cost": finance_data["estimated_order_cost"],
    "allocated_order_qty": finance_data["allocated_order_qty"],
    "allocated_order_cost": finance_data["allocated_order_cost"],
    "shortage_severity": finance_data["shortage_severity"],
    "approval_status": inventory_data["approval_status"]
}

print("--- VERIFIED BUSINESS FACTS ---")

for key, value in verified_facts.items():
    print(f"{key}: {value}")

# Verify consistency and purchasing safeguards
assert verified_facts["recommended_order_qty"] == 744
assert verified_facts["allocated_order_qty"] == 0
assert verified_facts["allocated_order_cost"] == 0
assert verified_facts["approval_status"] == "PENDING"

print("\nEvidence consistency: PASSED")
print("Purchasing approval still required: True")


--- VERIFIED BUSINESS FACTS ---
item_id: FOODS_3_090
recommended_order_qty: 744
estimated_order_cost: 14560.08
allocated_order_qty: 0
allocated_order_cost: 0.0
shortage_severity: MODERATE
approval_status: PENDING

Evidence consistency: PASSED
Purchasing approval still required: True



### 39. Detect Misleading Statements in AI Recommendations


In [44]:

def check_ai_funding_statement(answer, facts):

    issues = []
    answer_lower = answer.lower()

    # Check for misleading missing-funding statements
    if facts["allocated_order_cost"] == 0:

        if "not specified" in answer_lower:
            issues.append(
                "AI incorrectly describes known zero funding "
                "as unspecified."
            )

    return {
        "passed": len(issues) == 0,
        "issues": issues,
        "verified_allocated_cost": facts["allocated_order_cost"],
        "verified_allocated_qty": facts["allocated_order_qty"]
    }


# Check our previous AI recommendation
funding_check = check_ai_funding_statement(
    corrected_answer,
    verified_facts
)

print("Funding statement check:", 
      "PASSED" if funding_check["passed"] else "FLAGGED")

print("Issues:", funding_check["issues"])
print("Verified allocated cost:",
      funding_check["verified_allocated_cost"])
print("Verified allocated quantity:",
      funding_check["verified_allocated_qty"])


Funding statement check: FLAGGED
Issues: ['AI incorrectly describes known zero funding as unspecified.']
Verified allocated cost: 0.0
Verified allocated quantity: 0



### 40. Generate a Verified Business Decision Report


In [45]:

def generate_verified_manager_report(facts):

    report = f"""
BUSINESS DECISION REPORT
------------------------
Product: {facts['item_id']}

INVENTORY
Recommended reorder quantity: {facts['recommended_order_qty']} units

FINANCIAL ANALYSIS
Estimated order cost: ${facts['estimated_order_cost']:,.2f}
Allocated order quantity: {facts['allocated_order_qty']} units
Allocated order cost: ${facts['allocated_order_cost']:,.2f}

BUSINESS RISK
Shortage severity: {facts['shortage_severity']}

MANAGEMENT DECISION
Approval status: {facts['approval_status']}
Purchase executed: No

RECOMMENDATION
Review the proposed order, available funding,
and supplier delivery arrangements before approval.

Note: Inventory and financial inputs are simulated.
"""
    return report.strip()


verified_report = generate_verified_manager_report(
    verified_facts
)

print(verified_report)


BUSINESS DECISION REPORT
------------------------
Product: FOODS_3_090

INVENTORY
Recommended reorder quantity: 744 units

FINANCIAL ANALYSIS
Estimated order cost: $14,560.08
Allocated order quantity: 0 units
Allocated order cost: $0.00

BUSINESS RISK
Shortage severity: MODERATE

MANAGEMENT DECISION
Approval status: PENDING
Purchase executed: No

RECOMMENDATION
Review the proposed order, available funding,
and supplier delivery arrangements before approval.

Note: Inventory and financial inputs are simulated.



### 41. Validate Evidence Before Releasing Manager Reports


In [46]:

def release_manager_report(evidence):

    required = ["inventory", "finance"]

    # Check that both evidence sources exist
    missing = [
        source for source in required
        if source not in evidence
    ]

    if missing:
        return {
            "status": "BLOCKED",
            "reason": f"Missing evidence: {missing}",
            "report": None
        }

    inventory = evidence["inventory"]
    finance = evidence["finance"]

    # Check that the data belongs to the same product
    if inventory["item_id"] != finance["item_id"]:
        return {
            "status": "BLOCKED",
            "reason": "Product IDs do not match.",
            "report": None
        }

    # Build the report from verified evidence
    facts = {
        "item_id": inventory["item_id"],
        "recommended_order_qty": inventory["recommended_order_qty"],
        "estimated_order_cost": finance["estimated_order_cost"],
        "allocated_order_qty": finance["allocated_order_qty"],
        "allocated_order_cost": finance["allocated_order_cost"],
        "shortage_severity": finance["shortage_severity"],
        "approval_status": inventory["approval_status"]
    }

    return {
        "status": "READY_FOR_MANAGER_REVIEW",
        "report": generate_verified_manager_report(facts),
        "purchase_executed": False
    }


# Test 1: Complete evidence
complete_test = release_manager_report(
    evidence_package
)

# Test 2: Missing inventory evidence
incomplete_test = release_manager_report({
    "finance": evidence_package["finance"]
})

print("Complete evidence:", complete_test["status"])
print("Incomplete evidence:", incomplete_test["status"])
print("Blocked reason:", incomplete_test["reason"])
print("Purchase executed:", complete_test["purchase_executed"])

assert complete_test["status"] == "READY_FOR_MANAGER_REVIEW"
assert incomplete_test["status"] == "BLOCKED"

print("Validation gate tests: PASSED")


Complete evidence: READY_FOR_MANAGER_REVIEW
Incomplete evidence: BLOCKED
Blocked reason: Missing evidence: ['inventory']
Purchase executed: False
Validation gate tests: PASSED



### 42. Integrate Agent Execution and Verified Reporting


In [47]:

def run_verified_business_agent(question, item_id):

    # 1. Run the real local LLM agent
    agent = run_ollama_agent(question)

    # 2. Retrieve required evidence through approved tools
    evidence = {}
    required = {
        "inventory": "inventory_lookup",
        "finance": "business_risk_lookup"
    }

    for source, tool_name in required.items():

        execution = execute_business_tool({
            "name": tool_name,
            "arguments": json.dumps({"item_id": item_id})
        })

        if execution["status"] != "SUCCESS":
            return {
                "status": "BLOCKED",
                "reason": f"{tool_name} failed.",
                "purchase_executed": False
            }

        evidence[source] = execution["result"]

    # 3. Check required data fields
    required_fields = {
        "inventory": [
            "item_id",
            "recommended_order_qty",
            "approval_status"
        ],
        "finance": [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    }

    for source, fields in required_fields.items():
        if any(field not in evidence[source] for field in fields):
            return {
                "status": "BLOCKED",
                "reason": f"Incomplete {source} evidence.",
                "purchase_executed": False
            }

    # 4. Reject evidence for the wrong product
    if any(
        evidence[source]["item_id"] != item_id
        for source in required
    ):
        return {
            "status": "BLOCKED",
            "reason": "Product ID mismatch.",
            "purchase_executed": False
        }

    # 5. Release the verified report
    report = release_manager_report(evidence)

    return {
        "agent_status": agent["status"],
        "agent_tools": [
            entry["tool"] for entry in agent["execution_log"]
        ],
        "report_status": report["status"],
        "verified_report": report["report"],
        "purchase_executed": False
    }


# Test the integrated workflow
integrated_result = run_verified_business_agent(
    "For product FOODS_3_090, how much should "
    "we reorder, what will it cost, and what "
    "risks should the manager review?",
    "FOODS_3_090"
)

print("\n--- INTEGRATED AGENT RESULT ---")
print("Agent status:", integrated_result.get("agent_status"))
print("LLM tools used:", integrated_result.get("agent_tools"))
print("Report status:", integrated_result.get("report_status"))

print("\n--- VERIFIED REPORT ---")
print(integrated_result.get("verified_report"))

print("\nPurchase executed:",
      integrated_result["purchase_executed"])


Round 1: 1 tool(s) executed

--- INTEGRATED AGENT RESULT ---
Agent status: COMPLETED
LLM tools used: ['business_risk_lookup']
Report status: READY_FOR_MANAGER_REVIEW

--- VERIFIED REPORT ---
BUSINESS DECISION REPORT
------------------------
Product: FOODS_3_090

INVENTORY
Recommended reorder quantity: 744 units

FINANCIAL ANALYSIS
Estimated order cost: $14,560.08
Allocated order quantity: 0 units
Allocated order cost: $0.00

BUSINESS RISK
Shortage severity: MODERATE

MANAGEMENT DECISION
Approval status: PENDING
Purchase executed: No

RECOMMENDATION
Review the proposed order, available funding,
and supplier delivery arrangements before approval.

Note: Inventory and financial inputs are simulated.

Purchase executed: False



### 43. Record Tool Evidence in the Agent Execution Log


In [48]:

# Test whether the agent now records actual evidence
agent_evidence_test = run_ollama_agent(
    "What is the estimated purchasing cost "
    "for product FOODS_3_090?"
)

print("\n--- AGENT EVIDENCE LOG ---")

for entry in agent_evidence_test["execution_log"]:
    print("Tool:", entry["tool"])
    print("Status:", entry["status"])
    print("Evidence:", entry["result"])

print(
    "\nEvidence recorded:",
    any(
        entry["result"] is not None
        for entry in agent_evidence_test["execution_log"]
    )
)


Round 1: 1 tool(s) executed

--- AGENT EVIDENCE LOG ---
Tool: business_risk_lookup
Status: SUCCESS
Evidence: {'item_id': 'FOODS_3_090', 'estimated_order_cost': 14560.08, 'allocated_order_qty': 0, 'allocated_order_cost': 0.0, 'shortage_severity': 'MODERATE', 'management_action': 'REVIEW_DELIVERY_SCHEDULE', 'decision_status': 'PENDING'}

Evidence recorded: True



### 44. Reuse Agent Evidence and Recover Missing Data


In [49]:

def collect_agent_evidence(agent_result, item_id):

    required_tools = {
        "inventory_lookup": "inventory",
        "business_risk_lookup": "finance"
    }

    evidence = {}
    recovered_tools = []

    # Reuse successful results already collected by the LLM
    for entry in agent_result["execution_log"]:

        tool_name = entry["tool"]

        if (
            tool_name in required_tools
            and entry["status"] == "SUCCESS"
            and isinstance(entry.get("result"), dict)
            and entry["result"].get("item_id") == item_id
        ):
            source = required_tools[tool_name]
            evidence[source] = entry["result"]

    reused_sources = list(evidence.keys())

    # Retrieve only missing evidence
    for tool_name, source in required_tools.items():

        if source in evidence:
            continue

        execution = execute_business_tool({
            "name": tool_name,
            "arguments": json.dumps({"item_id": item_id})
        })

        if (
            execution["status"] != "SUCCESS"
            or not isinstance(execution.get("result"), dict)
            or execution["result"].get("item_id") != item_id
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Unable to recover {source} evidence.",
                "evidence": evidence
            }

        evidence[source] = execution["result"]
        recovered_tools.append(tool_name)

    return {
        "status": "EVIDENCE_COLLECTED",
        "evidence": evidence,
        "reused_sources": reused_sources,
        "recovered_tools": recovered_tools
    }


# Use the agent result from Step 43
evidence_result = collect_agent_evidence(
    agent_evidence_test,
    "FOODS_3_090"
)

print("Status:", evidence_result["status"])
print("Reused evidence:", evidence_result.get("reused_sources"))
print("Recovered tools:", evidence_result.get("recovered_tools"))
print("Evidence sources:", list(evidence_result["evidence"].keys()))

# Check the report gate
if evidence_result["status"] == "EVIDENCE_COLLECTED":
    checked_report = release_manager_report(
        evidence_result["evidence"]
    )
    print("Report status:", checked_report["status"])
else:
    print("Report status: BLOCKED")


Status: EVIDENCE_COLLECTED
Reused evidence: ['finance']
Recovered tools: ['inventory_lookup']
Evidence sources: ['finance', 'inventory']
Report status: READY_FOR_MANAGER_REVIEW



### 45. Build an Evidence-Aware Agent Workflow


In [50]:

def run_evidence_aware_agent(question, item_id):

    # Step 1: Run our actual local LLM agent
    agent = run_ollama_agent(question)

    # Step 2: Collect and reuse tool evidence
    evidence_result = collect_agent_evidence(
        agent,
        item_id
    )

    # Stop if evidence cannot be recovered
    if evidence_result["status"] != "EVIDENCE_COLLECTED":
        return {
            "status": "BLOCKED",
            "reason": evidence_result.get("reason"),
            "purchase_executed": False
        }

    # Step 3: Validate evidence before releasing report
    report_result = release_manager_report(
        evidence_result["evidence"]
    )

    return {
        "status": report_result["status"],
        "agent_status": agent["status"],
        "agent_tools": [
            entry["tool"]
            for entry in agent["execution_log"]
        ],
        "reused_sources": evidence_result["reused_sources"],
        "recovered_tools": evidence_result["recovered_tools"],
        "report": report_result["report"],
        "purchase_executed": False
    }


# Test our improved workflow
final_agent_result = run_evidence_aware_agent(
    "For product FOODS_3_090, how much should "
    "we reorder, what will it cost, and what "
    "risks should the manager review?",
    "FOODS_3_090"
)

print("\n--- EVIDENCE-AWARE AGENT ---")
print("Status:", final_agent_result["status"])
print("Agent status:", final_agent_result["agent_status"])
print("LLM-selected tools:", final_agent_result["agent_tools"])
print("Reused evidence:", final_agent_result["reused_sources"])
print("Recovered tools:", final_agent_result["recovered_tools"])

print("\n--- MANAGER REPORT ---")
print(final_agent_result["report"])

print("\nPurchase executed:",
      final_agent_result["purchase_executed"])


Round 1: 1 tool(s) executed

--- EVIDENCE-AWARE AGENT ---
Status: READY_FOR_MANAGER_REVIEW
Agent status: COMPLETED
LLM-selected tools: ['business_risk_lookup']
Reused evidence: ['finance']
Recovered tools: ['inventory_lookup']

--- MANAGER REPORT ---
BUSINESS DECISION REPORT
------------------------
Product: FOODS_3_090

INVENTORY
Recommended reorder quantity: 744 units

FINANCIAL ANALYSIS
Estimated order cost: $14,560.08
Allocated order quantity: 0 units
Allocated order cost: $0.00

BUSINESS RISK
Shortage severity: MODERATE

MANAGEMENT DECISION
Approval status: PENDING
Purchase executed: No

RECOMMENDATION
Review the proposed order, available funding,
and supplier delivery arrangements before approval.

Note: Inventory and financial inputs are simulated.

Purchase executed: False



### 46. Test Agent Generalization Across Products


In [51]:

# Select another product from our actual recommendations
other_products = inventory_recommendations[
    inventory_recommendations["item_id"] != "FOODS_3_090"
]

assert not other_products.empty, "No additional products available."

test_product = other_products.iloc[0]["item_id"]

print("Testing product:", test_product)

# Run our LLM-based evidence-aware workflow
second_agent_result = run_evidence_aware_agent(
    f"For product {test_product}, how many units "
    "should we reorder, what will it cost, "
    "and what risks should the manager review?",
    test_product
)

print("\n--- SECOND PRODUCT TEST ---")
print("Report status:", second_agent_result["status"])
print("Agent status:", second_agent_result["agent_status"])
print("LLM tools:", second_agent_result["agent_tools"])
print("Recovered tools:", second_agent_result["recovered_tools"])

print("\n--- MANAGER REPORT ---")
print(second_agent_result["report"])

# Check the result
assert second_agent_result["status"] == "READY_FOR_MANAGER_REVIEW"
assert f"Product: {test_product}" in second_agent_result["report"]
assert second_agent_result["purchase_executed"] is False

print("\nSecond product test: PASSED")


Testing product: FOODS_3_252
Round 1: 1 tool(s) executed

--- SECOND PRODUCT TEST ---
Report status: READY_FOR_MANAGER_REVIEW
Agent status: COMPLETED
LLM tools: ['business_risk_lookup']
Recovered tools: ['inventory_lookup']

--- MANAGER REPORT ---
BUSINESS DECISION REPORT
------------------------
Product: FOODS_3_252

INVENTORY
Recommended reorder quantity: 735 units

FINANCIAL ANALYSIS
Estimated order cost: $8,474.55
Allocated order quantity: 735 units
Allocated order cost: $8,474.55

BUSINESS RISK
Shortage severity: SEVERE

MANAGEMENT DECISION
Approval status: PENDING
Purchase executed: No

RECOMMENDATION
Review the proposed order, available funding,
and supplier delivery arrangements before approval.

Note: Inventory and financial inputs are simulated.

Second product test: PASSED



### 47. Test Unknown Product Handling


In [52]:

import json
from pprint import pprint

unknown_product = "DRAGON_CARROT_999"

# Make sure this product is not in our data
assert unknown_product not in inventory_recommendations["item_id"].values
assert unknown_product not in business_decisions["item_id"].values

# Test inventory and finance tools
for tool_name in ["inventory_lookup", "business_risk_lookup"]:

    execution = execute_business_tool({
        "name": tool_name,
        "arguments": json.dumps({
            "item_id": unknown_product
        })
    })

    print("\nTool:", tool_name)
    print("Execution status:", execution["status"])
    print("Returned information:")
    pprint(execution.get("result", execution.get("reason")))

    print("-" * 40)



Tool: inventory_lookup
Execution status: SUCCESS
Returned information:
{'item_id': 'DRAGON_CARROT_999',
 'message': 'No pending reorder recommendation found.',
 'status': 'NOT_FOUND'}
----------------------------------------

Tool: business_risk_lookup
Execution status: SUCCESS
Returned information:
{'item_id': 'DRAGON_CARROT_999',
 'message': 'No business decision record found.',
 'status': 'NOT_FOUND'}
----------------------------------------



### 48. Strengthen Manager Report Evidence Validation


In [53]:

import json

# ==================================================
# STEP 48: VALIDATE BUSINESS EVIDENCE
# ==================================================

def release_manager_report(evidence):

    # 1. Define the required fields
    required_fields = {
        "inventory": [
            "item_id",
            "recommended_order_qty",
            "approval_status"
        ],
        "finance": [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    }

    # 2. Validate each evidence source
    for source, fields in required_fields.items():

        if source not in evidence:
            return {
                "status": "BLOCKED",
                "reason": f"Missing {source} evidence.",
                "report": None
            }

        data = evidence[source]

        # Reject invalid or unknown product records
        if (
            not isinstance(data, dict)
            or data.get("status") == "NOT_FOUND"
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Invalid {source} record.",
                "report": None
            }

        # Check that all required fields exist
        if any(
            field not in data or data[field] is None
            for field in fields
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Incomplete {source} record.",
                "report": None
            }

    # 3. Extract verified evidence
    inventory = evidence["inventory"]
    finance = evidence["finance"]

    # Check product ID consistency
    if inventory["item_id"] != finance["item_id"]:
        return {
            "status": "BLOCKED",
            "reason": "Product IDs do not match.",
            "report": None
        }

    # 4. Build verified facts
    facts = {
        "item_id": inventory["item_id"],
        "recommended_order_qty": inventory["recommended_order_qty"],
        "estimated_order_cost": finance["estimated_order_cost"],
        "allocated_order_qty": finance["allocated_order_qty"],
        "allocated_order_cost": finance["allocated_order_cost"],
        "shortage_severity": finance["shortage_severity"],
        "approval_status": inventory["approval_status"]
    }

    # 5. Generate the report
    return {
        "status": "READY_FOR_MANAGER_REVIEW",
        "report": generate_verified_manager_report(facts),
        "purchase_executed": False
    }


# ==================================================
# TEST 1: KNOWN PRODUCT
# ==================================================

valid_test = release_manager_report(
    evidence_package
)


# ==================================================
# TEST 2: UNKNOWN PRODUCT
# ==================================================

unknown_product = "DRAGON_CARROT_999"

# Retrieve unknown inventory record
unknown_inventory = execute_business_tool({
    "name": "inventory_lookup",
    "arguments": json.dumps({
        "item_id": unknown_product
    })
})

# Retrieve unknown financial record
unknown_finance = execute_business_tool({
    "name": "business_risk_lookup",
    "arguments": json.dumps({
        "item_id": unknown_product
    })
})

# Validate unknown product evidence
invalid_test = release_manager_report({
    "inventory": unknown_inventory["result"],
    "finance": unknown_finance["result"]
})


# ==================================================
# DISPLAY RESULTS
# ==================================================

print("Known product:", valid_test["status"])
print("Unknown product:", invalid_test["status"])
print("Blocked reason:", invalid_test["reason"])

# Verify both tests
assert valid_test["status"] == "READY_FOR_MANAGER_REVIEW"
assert invalid_test["status"] == "BLOCKED"
assert invalid_test["report"] is None

print("Unknown product safety test: PASSED")


Known product: READY_FOR_MANAGER_REVIEW
Unknown product: BLOCKED
Blocked reason: Invalid inventory record.
Unknown product safety test: PASSED



### 49. Validate Business Numerical Values


In [57]:

import copy
import math
from numbers import Real, Integral

def validate_business_numbers(evidence):

    # First check the existing report validation
    basic_check = basic_release_manager_report(evidence)

    if basic_check["status"] == "BLOCKED":
        return basic_check

    inventory = evidence["inventory"]
    finance = evidence["finance"]

    # Check quantities
    quantity_fields = [
        ("inventory", inventory, "recommended_order_qty"),
        ("finance", finance, "allocated_order_qty")
    ]

    for source, data, field in quantity_fields:
        value = data[field]

        if (
            not isinstance(value, Integral)
            or isinstance(value, bool)
            or value < 0
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Invalid {source} quantity: {field}"
            }

    # Check financial values
    for field in ["estimated_order_cost", "allocated_order_cost"]:
        value = finance[field]

        if (
            not isinstance(value, Real)
            or isinstance(value, bool)
            or not math.isfinite(value)
            or value < 0
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Invalid financial value: {field}"
            }

    # Funding quantity cannot exceed proposed quantity
    if (
        finance["allocated_order_qty"]
        > inventory["recommended_order_qty"]
    ):
        return {
            "status": "BLOCKED",
            "reason": "Funded quantity exceeds recommended quantity."
        }

    return {
        "status": "PASSED",
        "reason": "Numerical business evidence is valid."
    }


# Test 1: Valid values
valid_numbers = validate_business_numbers(
    evidence_package
)

# Test 2: Negative quantity
negative_evidence = copy.deepcopy(evidence_package)
negative_evidence["inventory"]["recommended_order_qty"] = -50

negative_test = validate_business_numbers(
    negative_evidence
)

# Test 3: Negative cost
negative_cost_evidence = copy.deepcopy(evidence_package)
negative_cost_evidence["finance"]["estimated_order_cost"] = -100

negative_cost_test = validate_business_numbers(
    negative_cost_evidence
)

print("Valid numbers:", valid_numbers["status"])
print("Negative quantity:", negative_test["status"])
print("Negative cost:", negative_cost_test["status"])

assert valid_numbers["status"] == "PASSED"
assert negative_test["status"] == "BLOCKED"
assert negative_cost_test["status"] == "BLOCKED"

print("Numerical validation tests: PASSED")


Valid numbers: PASSED
Negative quantity: BLOCKED
Negative cost: BLOCKED
Numerical validation tests: PASSED



### 50. Integrate Numerical Validation into Report Release


In [60]:

import math
from numbers import Real, Integral

def release_manager_report(evidence):

    required_fields = {
        "inventory": [
            "item_id",
            "recommended_order_qty",
            "approval_status"
        ],
        "finance": [
            "item_id",
            "estimated_order_cost",
            "allocated_order_qty",
            "allocated_order_cost",
            "shortage_severity"
        ]
    }

    # 1. Check evidence sources and required fields
    for source, fields in required_fields.items():

        data = evidence.get(source)

        if not isinstance(data, dict) or data.get("status") == "NOT_FOUND":
            return {
                "status": "BLOCKED",
                "reason": f"Missing or invalid {source} evidence.",
                "report": None,
                "purchase_executed": False
            }

        if any(field not in data or data[field] is None for field in fields):
            return {
                "status": "BLOCKED",
                "reason": f"Incomplete {source} evidence.",
                "report": None,
                "purchase_executed": False
            }

    inventory = evidence["inventory"]
    finance = evidence["finance"]

    # 2. Check product ID consistency
    if (
        not isinstance(inventory["item_id"], str)
        or not inventory["item_id"].strip()
        or inventory["item_id"] != finance["item_id"]
    ):
        return {
            "status": "BLOCKED",
            "reason": "Invalid or mismatched product IDs.",
            "report": None,
            "purchase_executed": False
        }

    # 3. Validate integer quantities
    for data, field in [
        (inventory, "recommended_order_qty"),
        (finance, "allocated_order_qty")
    ]:
        value = data[field]

        if (
            not isinstance(value, Integral)
            or isinstance(value, bool)
            or value < 0
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Invalid quantity: {field}",
                "report": None,
                "purchase_executed": False
            }

    # 4. Validate financial amounts
    for field in ["estimated_order_cost", "allocated_order_cost"]:
        value = finance[field]

        if (
            not isinstance(value, Real)
            or isinstance(value, bool)
            or not math.isfinite(value)
            or value < 0
        ):
            return {
                "status": "BLOCKED",
                "reason": f"Invalid financial value: {field}",
                "report": None,
                "purchase_executed": False
            }

    # 5. Check funding quantity consistency
    if finance["allocated_order_qty"] > inventory["recommended_order_qty"]:
        return {
            "status": "BLOCKED",
            "reason": "Funded quantity exceeds recommended quantity.",
            "report": None,
            "purchase_executed": False
        }

    # 6. Create the verified report
    facts = {
        "item_id": inventory["item_id"],
        "recommended_order_qty": inventory["recommended_order_qty"],
        "estimated_order_cost": finance["estimated_order_cost"],
        "allocated_order_qty": finance["allocated_order_qty"],
        "allocated_order_cost": finance["allocated_order_cost"],
        "shortage_severity": finance["shortage_severity"],
        "approval_status": inventory["approval_status"]
    }

    return {
        "status": "READY_FOR_MANAGER_REVIEW",
        "report": generate_verified_manager_report(facts),
        "purchase_executed": False
    }


In [61]:

# Test 1: Valid evidence
valid_report = release_manager_report(evidence_package)

# Test 2: Negative quantity
invalid_report = release_manager_report(negative_evidence)

# Test 3: Unknown product
unknown_report = release_manager_report({
    "inventory": unknown_inventory["result"],
    "finance": unknown_finance["result"]
})

print("Valid report:", valid_report["status"])
print("Negative quantity:", invalid_report["status"])
print("Unknown product:", unknown_report["status"])

assert valid_report["status"] == "READY_FOR_MANAGER_REVIEW"
assert invalid_report["status"] == "BLOCKED"
assert unknown_report["status"] == "BLOCKED"

print("Step 50 validation: PASSED")


Valid report: READY_FOR_MANAGER_REVIEW
Negative quantity: BLOCKED
Unknown product: BLOCKED
Step 50 validation: PASSED



### 51. Run Final Integrated Agent Safety Test


In [62]:

# Run our integrated local LLM agent
final_safety_test = run_evidence_aware_agent(
    "For product FOODS_3_090, what quantity should "
    "we reorder, what is the estimated cost, and "
    "what risks should the manager review?",
    "FOODS_3_090"
)

# Get the validated report
report_text = final_safety_test.get("report") or ""

# Check the final workflow
checks = {
    "Manager report ready": (
        final_safety_test["status"]
        == "READY_FOR_MANAGER_REVIEW"
    ),
    "Correct product": "FOODS_3_090" in report_text,
    "Correct quantity": (
        "Recommended reorder quantity: 744 units"
        in report_text
    ),
    "Correct cost": "$14,560.08" in report_text,
    "Approval pending": (
        "Approval status: PENDING" in report_text
    ),
    "No purchase executed": (
        final_safety_test["purchase_executed"] is False
    )
}

print("\n--- FINAL SAFETY CHECKS ---")

for name, passed in checks.items():
    print(f"{name}: {'PASSED' if passed else 'FAILED'}")

print("\nOverall result:",
      "PASSED" if all(checks.values()) else "FAILED")


Round 1: 1 tool(s) executed

--- FINAL SAFETY CHECKS ---
Manager report ready: PASSED
Correct product: PASSED
Correct quantity: PASSED
Correct cost: PASSED
Approval pending: PASSED
No purchase executed: PASSED

Overall result: PASSED



### 52. Save Final Agent Integration Test Results


In [63]:

import json
from pathlib import Path
from datetime import datetime, timezone

# Create a folder for evaluation results
evaluation_dir = Path("../data/processed/agent_evaluation")
evaluation_dir.mkdir(parents=True, exist_ok=True)

# Prepare the test summary
evaluation_summary = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "notebook": "06_agentic_ai_workflow",
    "model": "llama3.2:latest",
    "product": "FOODS_3_090",
    "test_type": "integration_smoke_test",
    "checks": checks,
    "passed_checks": sum(checks.values()),
    "total_checks": len(checks),
    "overall_passed": all(checks.values()),
    "purchase_executed": False
}

# Save the results
evaluation_path = evaluation_dir / "nb6_integration_test.json"

with open(evaluation_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_summary, f, indent=4)

# Reload to verify
with open(evaluation_path, "r", encoding="utf-8") as f:
    saved_evaluation = json.load(f)

assert saved_evaluation == evaluation_summary

print("Evaluation saved:", evaluation_path.exists())
print("Model:", saved_evaluation["model"])
print(
    "Checks passed:",
    f"{saved_evaluation['passed_checks']}/{saved_evaluation['total_checks']}"
)
print("Overall result:",
      "PASSED" if saved_evaluation["overall_passed"] else "FAILED")
print("File verification: PASSED")


Evaluation saved: True
Model: llama3.2:latest
Checks passed: 6/6
Overall result: PASSED
File verification: PASSED



## Conclusion: Agentic AI Workflow and Tool Integration

### Project Objective

Develop a local LLM-powered retail decision-support system that coordinates inventory planning, financial analysis, and business risk assessment.

### System Architecture

The system integrates:

1. **Demand Forecasting:** A trained LightGBM model predicts future retail demand.
2. **Inventory Planning:** Forecast results support reorder quantity recommendations.
3. **Financial Analysis:** Purchasing costs, budget allocation, and shortage risks are evaluated.
4. **LLM Coordination:** Llama 3.2, running locally through Ollama, selects and invokes registered business tools.
5. **Evidence Validation:** Python checks business records, numerical values, and evidence completeness.
6. **Human Oversight:** Purchasing recommendations require manager approval.

### Key Achievements

- Registered five read-only business analysis tools.
- Demonstrated genuine LLM-driven tool selection.
- Successfully executed multiple LLM-selected tools.
- Implemented a bounded tool-calling agent loop.
- Added automatic recovery of missing business evidence.
- Developed evidence validation and report-release safeguards.
- Tested valid and unknown products.
- Rejected invalid purchasing quantities and costs.
- Generated verified manager reports.
- Saved integration test results.

### Evaluation Summary

The final integration smoke test passed all six predefined checks for product FOODS_3_090.

The agent also successfully generated a verified report for a second product, FOODS_3_252.

These results demonstrate basic functionality but do not establish reliability across the full dataset.

### Limitations

- The local Llama 3.2 model sometimes selects incomplete sets of business tools.
- Deterministic Python checks recover missing evidence.
- The system currently focuses on single-product purchasing decisions.
- Business inventory, supplier costs, and purchasing budgets are simulated.
- Forecasting errors may affect downstream purchasing recommendations.
- The system has not been evaluated comprehensively across diverse business scenarios.
- No real supplier purchases or financial transactions are executed.

### Next Steps

Notebook 07 will evaluate the agent's tool selection, evidence completeness, numerical accuracy, error handling, and purchasing safeguards across multiple scenarios.

A Streamlit application will subsequently provide an interactive demonstration of the system.

### Final Outcome

A functional hybrid Agentic AI prototype was developed using LightGBM forecasting, Python business tools, and a local Llama 3.2 language model.

The system demonstrates LLM-driven tool coordination with deterministic evidence validation and human-controlled purchasing decisions.
